# ⚖️ EasyPasal: LLM-Judged Evaluation on Free GPU (T3.3)

This notebook runs the **LLM-as-a-Judge evaluation harness** for Indonesian financial & banking regulations (*EasyPasal*) using free GPU resources on **Google Colab** (T4, 15GB VRAM) or **Kaggle** (P100/T4, 16GB VRAM).

### Evaluation Methodology:
- **Judge Model**: `Qwen/Qwen2.5-7B-Instruct` loaded in **4-bit NF4 quantization** via `bitsandbytes` (~5.5 GB VRAM footprint).
- **Faithfulness (RAGAS-style)**: Atomic claim extraction and entailment verification against retrieved context. Acceptance Gate: **$\ge 90\%$**.
- **Answer Correctness (DeepEval-style)**: Multi-fact True Positive / False Positive / False Negative alignment ($F_1$) against reference `gold_answer`. Guideline: **$\ge 80\%$**.
- **Legal Abstention Invariant**: Out-of-scope and adversarial queries must produce standardized refusal (*"Berdasarkan regulasi yang tersedia, tidak ditemukan dasar hukum..."*), receiving 100% faithfulness and strict refusal verification.

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(
        "WARNING: No GPU detected. Make sure to select 'T4 GPU' under Runtime -> Change runtime type (Colab) or Settings -> Accelerator -> GPU (Kaggle)."
    )

In [ ]:
# 2. Install Dependencies
!pip install -q transformers accelerate bitsandbytes httpx pandas pydantic

In [ ]:
# 3. Clone Repository or Set Working Directory
import sys
from pathlib import Path

# If not already inside easypasal repository
if not Path("eval").exists():
    if not Path("easypasal").exists():
        !git clone https://github.com/robitalhazmi/easypasal.git
        %cd easypasal
    else:
        %cd easypasal
        !git fetch origin && git checkout main && git pull origin main
else:
    # Already inside easypasal root, ensure latest branch commits
    !git fetch origin && git checkout main && git pull origin main

# Ensure repository root and src are in sys.path
repo_root = Path.cwd().resolve()
for p in [str(repo_root), str(repo_root / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Current working directory: {Path.cwd()}")
judge_file = Path.cwd() / "eval" / "metrics" / "judge.py"
print(f"Judge module exists: {judge_file.exists()} ({judge_file})")

In [ ]:
# 4. Initialize 4-bit Open-Weight Judge Backend
import sys
from pathlib import Path

# Safety check for sys.path
repo_root = Path.cwd().resolve()
for p in [str(repo_root), str(repo_root / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
from eval.metrics.judge import MockJudgeBackend, TransformersJudgeBackend

if torch.cuda.is_available():
    print("Initializing Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 precision...")
    judge_backend = TransformersJudgeBackend(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        load_in_4bit=True,
        device_map="auto",
    )
    print("Judge model ready!")
else:
    print("Using MockJudgeBackend (deterministic CPU fallback).")
    judge_backend = MockJudgeBackend()

In [ ]:
# 5. Load Evaluation Dataset and Index Context
from eval.run_judged_eval import load_chunk_text_map
from eval.split_eval import load_gold_items

dataset_path = "data/gold/dev.jsonl"
items = load_gold_items(dataset_path)
chunk_map = load_chunk_text_map("data/index/bm25/structure_aware_meta.json")

print(f"Loaded {len(items)} evaluation items from {dataset_path}")
print(f"Loaded {len(chunk_map)} chunk contexts from BM25 structure metadata")

In [ ]:
# 6. Run LLM-Judged Evaluation Runner
from eval.run_judged_eval import print_judged_report, run_judged_evaluation

backend_label = (
    "Transformers:Qwen/Qwen2.5-7B-Instruct-4bit" if torch.cuda.is_available() else "mock"
)

print("Starting evaluation run...")
report = run_judged_evaluation(
    backend=judge_backend,
    items=items,
    chunk_text_map=chunk_map,
    dataset_path=dataset_path,
    backend_name=backend_label,
)

print_judged_report(report)

In [ ]:
# 7. Save Evaluation Report & Export 35-Item Human Calibration Sample (for T3.4)
from eval.run_judged_eval import export_calibration_sample

out_json = report.save_json("data/eval_outputs/judged_eval_dev.json")
print(f"Saved full judged report to: {out_json}")

calib_path = export_calibration_sample(
    records=report.records,
    output_path="eval/judge_calibration/calibration_sample_35.jsonl",
    target_count=35,
)
print(f"Exported stratified calibration sample (35 items) to: {calib_path}")

In [ ]:
# 8. Tabulate Per-Category Results Table
import pandas as pd

rows = []
for cat, s in sorted(report.categories.items()):
    rows.append(
        {
            "Category": cat,
            "Total Items": s.total_items,
            "Answerable": s.answerable_items,
            "Mean Faithfulness": f"{s.mean_faithfulness:.1%}",
            "Mean Correctness": f"{s.mean_correctness:.1%}",
        }
    )

df_results = pd.DataFrame(rows)
display(df_results)